# FinSight Checkpoint 5 — Reproducibility & Results Review

This notebook inspects the existing Week 6 experiment-results export. It deliberately does not invent metrics or assume undocumented column names. Run from the repository root or adjust `RESULTS_PATH` below.

**Reminder:** model outputs are educational estimates, not investment advice.

In [2]:

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Locate the experiment results CSV
current_dir = Path.cwd()

possible_paths = [
    current_dir / "experiment_results.csv",
    current_dir / "notebooks" / "experiment_results.csv",
    current_dir.parent / "experiment_results.csv",
    current_dir.parent / "notebooks" / "experiment_results.csv",
]

RESULTS_PATH = next(
    (path for path in possible_paths if path.exists()),
    None
)

# Fallback: search nearby project directories
if RESULTS_PATH is None:
    search_roots = [current_dir, current_dir.parent]

    for root in search_roots:
        matches = list(root.glob("**/experiment_results.csv"))
        matches = [
            path for path in matches
            if ".venv" not in str(path)
            and "venv" not in str(path)
        ]

        if matches:
            RESULTS_PATH = matches[0]
            break

# Validate file availability
if RESULTS_PATH is None:
    raise FileNotFoundError(
        "experiment_results.csv was not found.\n"
        f"Current directory: {current_dir}\n"
        "Check the existing Week 6 experiment-results location."
    )

# Load actual experiment results
results = pd.read_csv(RESULTS_PATH)

print("=" * 55)
print("FINSIGHT CHECKPOINT 5 - RESULTS REVIEW")
print("=" * 55)

print(f"\nCSV Location: {RESULTS_PATH}")
print(f"Total Rows: {len(results):,}")
print(f"Total Columns: {len(results.columns)}")

print("\nDataset Preview:")
display(results.head())

print("\nColumn Information:")
display(
    pd.DataFrame({
        "Column": results.columns,
        "Data Type": results.dtypes.astype(str),
        "Missing Values": results.isnull().sum().values
    })
)

print("\nStatistical Summary:")
display(results.describe(include="all").T)

print("\nResults loaded successfully.")


FINSIGHT CHECKPOINT 5 - RESULTS REVIEW

CSV Location: C:\Users\siriy\Desktop\finsight-wealth-platform\notebooks\experiment_results.csv
Total Rows: 36
Total Columns: 12

Dataset Preview:


,profile,run,cov_method,ret_shrink,max_weight,ann_return,ann_vol,sharpe,max_drawdown,drawdown_ok,baseline_sharpe,avg_equity
0,conservative,0,sample,0.0,0.25,0.024099,0.055558,-0.646202,-0.024781,True,-0.867259,0.379721
1,conservative,1,sample,0.0,0.40,0.022199,0.057870,-0.653217,-0.025144,False,-0.867259,0.398803
2,conservative,2,sample,0.5,0.25,0.025055,0.055780,-0.626476,-0.024871,True,-0.867259,0.376813
3,conservative,3,sample,0.5,0.40,0.021003,0.056581,-0.689221,-0.025129,False,-0.867259,0.387409
4,conservative,4,sample,1.0,0.25,0.017051,0.057447,-0.747624,-0.036654,False,-0.867259,0.280000



Column Information:


,Column,Data Type,Missing Values
profile,profile,str,0
run,run,int64,0
cov_method,cov_method,str,0
ret_shrink,ret_shrink,float64,0
max_weight,max_weight,float64,0
ann_return,ann_return,float64,0
ann_vol,ann_vol,float64,0
sharpe,sharpe,float64,0
max_drawdown,max_drawdown,float64,0
drawdown_ok,drawdown_ok,bool,0



Statistical Summary:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
profile,36,3,conservative,12,NaN,NaN,NaN,NaN,NaN,NaN,NaN
run,36.0,NaN,NaN,NaN,5.5,3.50102,0.0,2.75,5.5,8.25,11.0
cov_method,36,2,sample,18,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ret_shrink,36.0,NaN,NaN,NaN,0.5,0.414039,0.0,0.0,0.5,1.0,1.0
max_weight,36.0,NaN,NaN,NaN,0.325,0.076064,0.25,0.25,0.325,0.4,0.4
ann_return,36.0,NaN,NaN,NaN,0.002474,0.045279,-0.160599,-0.015647,0.018571,0.031154,0.047584
ann_vol,36.0,NaN,NaN,NaN,0.083767,0.021735,0.055006,0.057764,0.087742,0.097866,0.118804
sharpe,36.0,NaN,NaN,NaN,-0.657968,0.416749,-2.188259,-0.751702,-0.635784,-0.328164,-0.225402
max_drawdown,36.0,NaN,NaN,NaN,-0.042931,0.016215,-0.082234,-0.055912,-0.038345,-0.031421,-0.020528
drawdown_ok,36,2,True,30,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Results loaded successfully.


## Numeric summary
Review the numeric experiment fields. Confirm the metric names and units against the experiment code before interpreting them.

In [ ]:
numeric = results.select_dtypes(include='number')
if numeric.empty:
    print('No numeric columns found in the experiment export.')
else:
    display(numeric.describe().T)

## Experiment metric distributions
The following plots are exploratory only. They display numeric columns present in the CSV; they do not label a column as Sharpe, return, or volatility unless that mapping has been verified in the source code.

In [ ]:
for column in numeric.columns:
    values = numeric[column].dropna()
    if values.empty:
        continue
    plt.figure(figsize=(7, 3.5))
    plt.hist(values, bins=min(20, max(5, values.nunique())))
    plt.title(f'Distribution: {column}')
    plt.xlabel(column)
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.show()

## Final review notes
- Verify the selected configuration against the MLflow experiment and source code.
- Distinguish in-sample expected metrics from walk-forward out-of-sample metrics.
- The repository README reports a negative mean out-of-sample Sharpe for the best configuration versus the equal-weight baseline. Retain this limitation in the model card and demo.
- Do not use this prototype to make real investment decisions.